# Arabic RAG Chatbot

- Dataset: Arabic Wikipedia — 30,000+ articles
- Splitters: Fixed / Recursive / Semantic
- Embedding: sentence-transformers (multilingual)
- Vector Store: FAISS
- LLM: OpenAI GPT-4o
- UI: Gradio

## Section 1: Install Libraries

In [ ]:
!pip -q install openai
!pip -q install gradio
!pip -q install faiss-cpu
!pip -q install sentence-transformers
!pip -q install datasets
!pip -q install langchain-text-splitters
!pip -q install scikit-learn

## Section 2: OpenAI API Key

In [ ]:
OPENAI_API_KEY = "sk-proj-..."  # put your key here

## Section 3: Load Arabic Dataset

Source: wikimedia/wikipedia — Arabic — 30,000+ articles

In [ ]:
from datasets import load_dataset

print("Loading Arabic Wikipedia dataset...")

ds = load_dataset(
    "wikimedia/wikipedia",
    "20231101.ar",
    split="train[:35000]"
)

texts = [row["text"] for row in ds if len(row["text"]) > 50]

print(f"Total texts loaded: {len(texts):,}")
print(f"30K requirement met: {len(texts) >= 30000}")
print("Sample text (first 200 chars):")
print(texts[0][:200])

## Section 4: Three Text Splitting Methods

| Method | Description |
|--------|-------------|
| Fixed | Splits by fixed character count |
| Recursive | Tries paragraphs, sentences, words in order |
| Semantic | Splits based on embedding similarity between sentences |

In [ ]:
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# Load embedding model
print("Loading embedding model...")
embed_model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
print("Embedding model loaded.")

# Use first 500 texts for chunking
sample_texts = texts[:500]


# 1. FIXED SIZE CHUNKING
def fixed_chunking(texts):
    splitter = CharacterTextSplitter(
        separator="\n",
        chunk_size=500,
        chunk_overlap=50,
    )
    chunks = []
    for text in texts:
        chunks.extend(splitter.split_text(text))
    return chunks

# 2. RECURSIVE CHUNKING

def recursive_chunking(texts):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=50,
        separators=["\n\n", "\n", ".", "\u060c", "\u061f", "!", " ", ""]
    )
    chunks = []
    for text in texts:
        chunks.extend(splitter.split_text(text))
    return chunks

# 3. SEMANTIC CHUNKING

def semantic_chunking(text, threshold=0.5):
    sentences = text.split(".")
    sentences = [s.strip() for s in sentences if s.strip()]
    if len(sentences) < 2:
        return [text]

    embeddings = embed_model.encode(sentences)
    chunks = []
    current_chunk = [sentences[0]]

    for i in range(1, len(sentences)):
        similarity = cosine_similarity([embeddings[i-1]], [embeddings[i]])[0][0]
        if similarity >= threshold:
            current_chunk.append(sentences[i])
        else:
            chunks.append(". ".join(current_chunk))
            current_chunk = [sentences[i]]

    chunks.append(". ".join(current_chunk))
    return chunks

def semantic_chunking_all(texts):
    chunks = []
    for text in texts[:100]:  # first 100 texts (semantic is slower)
        chunks.extend(semantic_chunking(text))
    return chunks

# Apply all three methods

print("Running Fixed chunking...")
fixed_chunks = fixed_chunking(sample_texts)
print(f"Fixed chunks count: {len(fixed_chunks):,}")

print("Running Recursive chunking...")
recursive_chunks = recursive_chunking(sample_texts)
print(f"Recursive chunks count: {len(recursive_chunks):,}")

print("Running Semantic chunking (this takes longer)...")
semantic_chunks = semantic_chunking_all(sample_texts)
print(f"Semantic chunks count: {len(semantic_chunks):,}")

print("\n" + "-" * 55)
print(f"{'Method':<15} {'Chunks':<15} {'Avg Length (chars)'}")
print("-" * 55)
for name, chunks in [("Fixed", fixed_chunks), ("Recursive", recursive_chunks), ("Semantic", semantic_chunks)]:
    avg = sum(len(c) for c in chunks) // max(len(chunks), 1)
    print(f"{name:<15} {len(chunks):<15,} {avg}")
print("-" * 55)

## Section 5: Build FAISS Vector Store

In [ ]:
import faiss
import numpy as np

def build_faiss_index(chunks, model):
    print(f"Building embeddings for {len(chunks):,} chunks...")
    embeddings = model.encode(chunks, batch_size=64, show_progress_bar=True)
    embeddings = np.array(embeddings).astype("float32")

    dim = embeddings.shape[1]
    index = faiss.IndexFlatL2(dim)
    index.add(embeddings)

    print(f"FAISS index built. Total vectors: {index.ntotal:,}")
    return index, chunks

def search_index(query, index, chunks, model, k=5):
    query_vec = model.encode([query]).astype("float32")
    distances, indices = index.search(query_vec, k)
    return [chunks[i] for i in indices[0]]

# Build index using recursive chunks (best for RAG)
print("Building FAISS vector store using recursive chunks...")
faiss_index, index_chunks = build_faiss_index(recursive_chunks, embed_model)

# Quick search test
print("Running search test...")
test_results = search_index(" الذكاء الاصطناعي ", faiss_index, index_chunks, embed_model)
print("Search test passed.")
print(f"Top result preview: {test_results[0][:200]}")

## Section 6: Build RAG Chain with OpenAI GPT-4o-mini

In [ ]:
from openai import OpenAI

client = OpenAI(api_key=OPENAI_API_KEY)

def chat_fn(user_message, history):

    # Step 1: Retrieve relevant chunks from FAISS
    docs = search_index(user_message, faiss_index, index_chunks, embed_model, k=5)
    context = "\n\n".join([f"[{i+1}] {doc}" for i, doc in enumerate(docs)])

    # Step 2: Build system prompt
    system_prompt = """You are an intelligent assistant that answers questions in Arabic.
Use only the information provided in the context below to answer.
If the answer is not in the context, say: 'No sufficient information available in the data.'
Always respond in clear, formal Arabic.

Context:
""" + context

    # Step 3: Build messages with conversation history
    messages = [{"role": "system", "content": system_prompt}]
    for human, assistant in history:
        messages.append({"role": "user", "content": human})
        messages.append({"role": "assistant", "content": assistant})
    messages.append({"role": "user", "content": user_message})

    # Step 4: Call OpenAI API
    completion = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        temperature=0.1,
        max_tokens=1024,
    )

    return completion.choices[0].message.content

# Quick test
print("Testing RAG chain...")
try:
    test_response = chat_fn("ما هو الذكاء الاصطناعي ؟", [])
    print("RAG chain test passed.")
    print(f"Response preview: {test_response[:300]}")
except Exception as e:
    print(f"ERROR: {e}")
    print("Check your OPENAI_API_KEY in Section 2")

## Section 7: Gradio UI

Note: Do NOT stop this cell or the interface will close.

In [ ]:
import gradio as gr

def user_submit(user_message, history):
    if not user_message.strip():
        return "", history
    try:
        response = chat_fn(user_message, history)
    except Exception as e:
        response = f"Error: {str(e)}"
    history = history + [(user_message, response)]
    return "", history

with gr.Blocks(title="Arabic RAG Chatbot") as demo:

    gr.Markdown("""
    <div style='text-align:center; direction:rtl'>
    <h1>Arabic RAG Chatbot</h1>
    <p>Powered by Arabic Wikipedia + FAISS + GPT-4o-mini</p>
    </div>
    """)

    chatbot = gr.Chatbot(
        label="Conversation",
        height=450,
        rtl=True,
        type="tuples"
    )

    msg = gr.Textbox(
        placeholder="Type your question in Arabic...",
        label="Your Question",
        rtl=True
    )

    clear = gr.Button("Clear")

    gr.Examples(
        examples=[
            "ما هي مكة المكرمة وما أهميتها؟",
            "من هو ابن سينا؟",
            "ما هي أهمية نهر النيل؟",
            "متى تأسست بغداد؟",
        ],
        inputs=msg,
        label="Examples"
    )

    state = gr.State([])

    msg.submit(user_submit, [msg, state], [msg, chatbot])
    clear.click(lambda: ([], ""), outputs=[chatbot, msg])

demo.launch(share=True)